# Predição de Depressão em Estudantes
## Projeto Final — Aprendizado de Máquina — DCOMP/UFS


## 1.1 Identificação e descrição do problema

**Título:** Predição de Depressão em Estudantes

**Integrantes:**
- Mateus (@Mat-Macedo)
- Luan (@luanorama)
- Evilyn (@EvilynAquino)

**Fonte dos dados:** [Student Depression Dataset — Kaggle](https://www.kaggle.com/datasets/hopesb/student-depression-dataset)

**Objetivo:** o projeto é tentar prever se um estudante tem indícios de depressão, usando informações como dados demográficos, vida acadêmica e hábitos do dia a dia. A ideia é que isso possa ajudar a identificar casos de risco mais cedo.

**Atributo-alvo:** `Depression` (0 = não, 1 = sim)

**Atributos preditivos:** idade, gênero, cidade, profissão, pressão acadêmica/trabalho, CGPA, satisfação com estudo/trabalho, duração do sono, hábitos alimentares, grau acadêmico, horas de estudo/trabalho, estresse financeiro, histórico familiar de doença mental, pensamentos suicidas prévios.

**Tipo da tarefa:** é uma classificação binária, porque o alvo só tem duas categorias possíveis (com ou sem indícios de depressão) e não um número contínuo.

> **Nota ética:** esse dataset fala de saúde mental e tem uma pergunta direta sobre pensamentos suicidas. A gente decidiu usar esse atributo como preditor, mas vamos discutir isso com mais cuidado lá na seção final, e não simplesmente tratar como se fosse uma variável qualquer.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")


In [ ]:
URL = "https://raw.githubusercontent.com/EvilynAquino/student-depression-prediction/main/data/student_depression_dataset.csv"

df = pd.read_csv(URL)
df.head()


## 1.2 Compreensão dos dados


In [ ]:
print(f"Registros: {df.shape[0]}")
print(f"Atributos: {df.shape[1]}")
df.info()


**O que deu pra perceber:** o dataset tem **27.901 linhas** e **18 colunas**. A maior parte das colunas numéricas (`Age`, `Academic Pressure`, `CGPA`, etc.) já vem como `float64`, e as colunas categóricas (`Gender`, `City`, `Sleep Duration`, etc.) vêm como texto (`object`). A coluna `id` é só um identificador, não serve pra nada na hora de treinar o modelo, então ela vai ser removida mais pra frente.


In [ ]:
df.isnull().sum().sort_values(ascending=False)


**O que deu pra perceber:** só tem **3 valores faltando**, e todos estão na coluna `Financial Stress`. É bem pouca coisa perto de quase 28 mil linhas (0,01%), então não atrapalha em nada a qualidade dos dados.


In [ ]:
print(f"Linhas duplicadas: {df.duplicated().sum()}")


**O que deu pra perceber:** não tem nenhuma linha duplicada no dataset.


In [ ]:
df['Depression'].value_counts()


In [ ]:
df['Depression'].value_counts(normalize=True).mul(100).round(2)


**O que deu pra perceber:** o alvo está **um pouco desbalanceado**: cerca de **58,5% dos estudantes (16.336)** têm indícios de depressão, e **41,5% (11.565)** não têm. Por causa disso, vamos usar estratificação na hora de separar treino e teste, e também tomar cuidado pra não usar só a acurácia como métrica de avaliação lá na frente.


**Inconsistências que apareceram:** tem valores `"Others"` tanto em `Sleep Duration` quanto em `Dietary Habits` (bem poucas vezes, menos de 20 cada), que não dizem nada específico sobre uma categoria.


In [ ]:
df['Sleep Duration'].value_counts()


In [ ]:
df['Dietary Habits'].value_counts()


In [ ]:
df['City'].value_counts().tail(25)

**Mais uma inconsistência que apareceu:** olhando as cidades com menos ocorrências, tem várias que não são cidade nenhuma — tipo `"Saanvi"`, `"Harsha"`, `"M.Tech"`, `"3.0"`, `"Less than 5 Kalyan"` e até a palavra `"City"` literalmente. Parece lixo de digitação ou de importação dos dados (algumas até parecem valor que vazou de outra coluna, tipo grau acadêmico ou nota). São só 26 linhas (0,09%), mas vamos tratar isso no pré-processamento, porque senão o modelo ia aprender essas categorias erradas como se fossem cidades de verdade.

## 1.3 Análise exploratória


In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
sns.countplot(data=df, x='Depression', ax=ax)
ax.set_xticklabels(['Sem depressão (0)', 'Com depressão (1)'])
ax.set_title('Distribuição do atributo-alvo')
plt.show()


**O que deu pra perceber:** o gráfico mostra bem o desbalanceamento que a gente comentou antes — a classe "com depressão" é maior, mas as duas têm quantidade suficiente pra treinar um modelo.


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(data=df, x='Age', hue='Depression', kde=True, bins=30, ax=ax)
ax.set_title('Distribuição de idade por classe de depressão')
plt.show()


**O que deu pra perceber:** a idade média dos estudantes com depressão é **24,9 anos**, e dos sem depressão é **27,1 anos** — a diferença é pequena, mas existe.


In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
sns.boxplot(data=df, x='Depression', y='Academic Pressure', ax=ax)
ax.set_xticklabels(['Sem depressão', 'Com depressão'])
ax.set_title('Pressão acadêmica por classe de depressão')
plt.show()


**O que deu pra perceber:** a pressão acadêmica média de quem tem depressão é **3,69**, e de quem não tem é **2,36** (numa escala de 0 a 5) — esse é um dos atributos que mais parece ter relação com o alvo.


In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
pd.crosstab(df['Have you ever had suicidal thoughts ?'], df['Depression'], normalize='index').mul(100).plot(
    kind='bar', stacked=True, ax=ax, color=['#4C72B0', '#DD8452']
)
ax.set_title('Depressão por histórico de pensamentos suicidas (%)')
ax.set_ylabel('% dentro do grupo')
ax.legend(['Sem depressão', 'Com depressão'])
plt.show()


**O que deu pra perceber:** essa foi a relação mais forte que apareceu na análise. Entre os estudantes que **já tiveram pensamentos suicidas**, **79% têm indícios de depressão**; já entre os que nunca tiveram, esse número cai pra **23%**. Isso levanta uma questão ética: será que o modelo não está só captando uma correlação meio óbvia clinicamente? A gente retoma essa discussão lá na seção 1.7.


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('id')
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Matriz de correlação entre variáveis numéricas')
plt.show()


**O que deu pra perceber:** `Academic Pressure`, `Financial Stress` e `Work/Study Hours` têm correlação positiva com `Depression`; `Study Satisfaction` tem correlação negativa. Já `Work Pressure` e `Job Satisfaction` têm correlação praticamente zero, porque quase todo mundo tem valor 0 nessas colunas.


In [ ]:
fig, ax = plt.subplots(figsize=(6,8))
df['City'].value_counts().head(15).plot(kind='barh', ax=ax)
ax.set_title('15 cidades mais frequentes no dataset')
ax.invert_yaxis()
plt.show()

print(f"Total de cidades diferentes: {df['City'].nunique()}")


**O que deu pra perceber:** o dataset tem **52 valores diferentes** na coluna `City`, mas nem todos são cidades de verdade (tem o lixo de digitação que a gente comentou lá em cima). Fora isso, tem bastante concentração em algumas cidades grandes. Essa quantidade grande de categorias vai ser tratada no pré-processamento com one-hot encoding, agrupando as menos frequentes (incluindo o lixo) numa categoria única, pra não gerar coluna esparsa à toa.

### Resumo da análise exploratória
- Alvo um pouco desbalanceado (58,5% / 41,5%) → precisa de estratificação.
- `Academic Pressure`, `Financial Stress`, `Work/Study Hours`: relação clara com a depressão.
- `Have you ever had suicidal thoughts ?`: o preditor mais forte, e também o mais delicado eticamente.
- `Work Pressure` e `Job Satisfaction`: quase sem variação, pouco útil pra prever.
- `City`: muitas categorias diferentes (52 no total).


## 1.4 Pré-processamento

Pra cada tratamento: qual foi o problema encontrado, o que foi feito, e por quê.

> Todos os ajustes (imputação, limites de outliers, encoding, escalonamento) são **calculados só com os dados de treino** e depois só aplicados no teste, pra não vazar informação de um conjunto pro outro.


## 1.5 Separação dos dados


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(['Depression', 'id'], axis=1)
y = df['Depression']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Treino: {X_train.shape[0]} linhas | Teste: {X_test.shape[0]} linhas")


**Justificativa:** usei a proporção 80/20, que é padrão pra dataset com dezenas de milhares de linhas (dá um teste com volume relevante sem tirar muito dado do treino). Usei estratificação pra manter a proporção 58,5%/41,5% de `Depression` nos dois conjuntos.


In [ ]:
from sklearn.impute import SimpleImputer

colunas_numericas = X_train.select_dtypes(include=['int64', 'float64']).columns
colunas_categoricas = X_train.select_dtypes(include=['object']).columns

imputer_num = SimpleImputer(strategy='mean')
imputer_cat = SimpleImputer(strategy='most_frequent')

X_train[colunas_numericas] = imputer_num.fit_transform(X_train[colunas_numericas])
X_train[colunas_categoricas] = imputer_cat.fit_transform(X_train[colunas_categoricas])

X_test[colunas_numericas] = imputer_num.transform(X_test[colunas_numericas])
X_test[colunas_categoricas] = imputer_cat.transform(X_test[colunas_categoricas])

print("Valores ausentes tratados. Nulos restantes:", X_train.isnull().sum().sum() + X_test.isnull().sum().sum())


**Problema:** 3 valores faltando em `Financial Stress`. **Tratamento:** imputação pela média (é coluna numérica). **Justificativa:** é muito pouca coisa (0,01%), então usar a média não distorce a distribuição original.


In [ ]:
Q1 = X_train[colunas_numericas].quantile(0.25)
Q3 = X_train[colunas_numericas].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

mascara_treino = ~((X_train[colunas_numericas] < limite_inferior) | (X_train[colunas_numericas] > limite_superior)).any(axis=1)
print(f"Outliers removidos do treino: {(~mascara_treino).sum()} ({(~mascara_treino).sum()/len(X_train)*100:.2f}%)")

X_train = X_train[mascara_treino]
y_train = y_train[mascara_treino]


**Problema:** valores muito fora da curva nos atributos numéricos (calculado pelo método IQR). **Tratamento:** remover as linhas que ficam fora dos limites, calculados só com o treino. **Justificativa:** só 17 linhas (0,08% do treino) foram afetadas — impacto bem pequeno na quantidade de dados, e evita que esses valores extremos atrapalhem o ajuste do modelo.


In [ ]:
# Agrupar cidades raras (incluindo os valores inconsistentes tipo "Saanvi", "3.0", etc.) numa categoria única
# O limite é calculado só com o treino, pra não vazar informação
contagem_cidades = X_train['City'].value_counts()
cidades_raras = contagem_cidades[contagem_cidades < 10].index

X_train['City'] = X_train['City'].replace(cidades_raras, 'Outras')
X_test['City'] = X_test['City'].replace(cidades_raras, 'Outras')

print(f"Cidades agrupadas em 'Outras': {len(cidades_raras)}")
print(f"Categorias de City restantes: {X_train['City'].nunique()}")

**Problema:** a coluna `City` tem 52 categorias, sendo que boa parte delas aparece muito pouco — inclusive os valores inconsistentes que a gente achou lá na seção 1.2 (`"Saanvi"`, `"3.0"`, etc.), que também são raros. **Tratamento:** agrupei todas as cidades que aparecem menos de 10 vezes no treino numa categoria só, `"Outras"`. **Justificativa:** isso resolve dois problemas de uma vez — reduz a quantidade de colunas que o one-hot encoding ia gerar (a maioria delas quase sem exemplo nenhum), e também dilui o lixo de digitação dentro de uma categoria genérica em vez de virar uma coluna própria enganosa. Como o agrupamento é feito só olhando a frequência no treino, não tem vazamento de informação do teste.

In [ ]:
X_train = pd.get_dummies(X_train, columns=colunas_categoricas, drop_first=True)
X_test = pd.get_dummies(X_test, columns=colunas_categoricas, drop_first=True)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print("Colunas finais após encoding:", X_train.shape[1])


**Problema:** variáveis categóricas (`Gender`, `City`, `Profession`, etc.) não dá pra usar direto em modelos de ML. **Tratamento:** one-hot encoding (`get_dummies`, com `drop_first=True` pra evitar multicolinearidade). **Justificativa:** os modelos usados aqui (SGDClassifier, RandomForest) não lidam com texto naturalmente; one-hot é o jeito padrão de tratar categóricas sem ordem. O `align` garante que treino e teste fiquem com as mesmas colunas, mesmo que alguma categoria rara apareça só em um dos dois conjuntos.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_escalonado = scaler.fit_transform(X_train)
X_test_escalonado = scaler.transform(X_test)

X_train = pd.DataFrame(X_train_escalonado, columns=X_train.columns)
X_test = pd.DataFrame(X_test_escalonado, columns=X_test.columns)

print("Pré-processamento concluído. Shapes finais:", X_train.shape, X_test.shape)


**Problema:** os atributos numéricos estão em escalas bem diferentes (por exemplo, `Age` em anos vs `CGPA` de 0 a 10). **Tratamento:** padronização com `StandardScaler` (média 0, desvio padrão 1). **Justificativa:** o SGDClassifier é sensível à escala das variáveis (ele usa gradiente descendente); sem escalonar, os atributos com valores maiores acabariam dominando o ajuste.


## 1.6 Modelagem

Aqui a gente compara um baseline com os dois modelos pedidos: **SGDClassifier** e **RandomForestClassifier**, usando validação cruzada estratificada (5 folds) em cima do conjunto de treino.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

baseline = DummyClassifier(strategy='most_frequent', random_state=42)
sgd = SGDClassifier(random_state=42, max_iter=1000)
rf = RandomForestClassifier(random_state=42, n_estimators=200)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
modelos = {'Baseline': baseline, 'SGDClassifier': sgd, 'RandomForest': rf}

resultados_cv = {}
for nome, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=skf, scoring='f1')
    resultados_cv[nome] = scores
    print(f"{nome}: F1 médio (CV) = {scores.mean():.4f} (+/- {scores.std():.4f})")


**Parâmetros principais:**
- `DummyClassifier(strategy='most_frequent')`: baseline que sempre chuta a classe majoritária (com depressão).
- `SGDClassifier(max_iter=1000)`: classificador linear que usa gradiente descendente estocástico.
- `RandomForestClassifier(n_estimators=200)`: floresta com 200 árvores de decisão.

**Resultado da validação cruzada (F1-score médio, 5 folds):**
- Baseline: **0,7388**
- SGDClassifier: **0,8587**
- RandomForest: **0,8693**

**Comparação:** os dois modelos ficaram bem acima do baseline (mais de 0,11 de F1 de diferença), o que mostra que os atributos realmente ajudam a prever depressão. O RandomForest teve o melhor resultado médio na validação cruzada, e também variou menos entre os folds (desvio padrão de 0,0029, contra 0,0050 do SGD).


In [ ]:
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
print("Modelos treinados.")


**Escolha do modelo final:** escolhi o **RandomForestClassifier** como modelo final. Além de ter tido o melhor F1 médio na validação cruzada, ele também deixa mais fácil ver a importância de cada atributo na decisão (coisa que o SGDClassifier, por ser linear, não mostra tão direto) — o que ajuda bastante na discussão da seção 1.7.


## 1.7 Avaliação e discussão


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

modelo_final = rf
y_pred = modelo_final.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Sem depressão', 'Com depressão']))

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Sem depressão', 'Com depressão'], ax=ax, cmap='Blues')
ax.set_title('Matriz de confusão — RandomForest (conjunto de teste)')
plt.show()


**Lendo a matriz de confusão e as métricas no conjunto de teste:**
- **Acurácia:** 84,0%
- **Precisão (classe "com depressão"):** 84,9%
- **Revocação (classe "com depressão"):** 88,3%
- **F1-score:** 86,6%

O modelo erra um pouco mais em **falsos positivos do que em falsos negativos**: de 2.313 estudantes sem depressão, 512 foram classificados errado como se tivessem depressão (falso positivo); de 3.268 estudantes com depressão, 381 foram classificados como se não tivessem (falso negativo). Pensando num cenário de triagem de saúde mental, o falso negativo é o erro mais preocupante (um estudante em risco passaria despercebido), então a revocação de 88,3% é um resultado bom, mas ainda deixa cerca de 1 em cada 8 casos reais sem detectar.


In [ ]:
importancias = pd.Series(modelo_final.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importancias.head(10).plot(kind='barh', figsize=(8,5))
plt.gca().invert_yaxis()
plt.title('Top 10 atributos mais importantes (RandomForest)')
plt.show()


**O que deu pra perceber:** o atributo mais importante pro modelo, de longe, é `Have you ever had suicidal thoughts ?_Yes` (quase 20% da importância total), seguido por `Academic Pressure` e `Financial Stress`. Isso bate com o que já tinha aparecido lá na análise exploratória (seção 1.3).

**Discussão:**

- **Qual modelo foi melhor e por quê:** o RandomForest teve o melhor F1 médio na validação cruzada (0,8693 contra 0,8587 do SGD), e também ganhou no teste (F1 de 0,8661, contra 0,8549 do SGD). Escolhi o RandomForest pela consistência nas duas avaliações e pela interpretabilidade extra (dá pra ver a importância dos atributos).

- **Que erros apareceram:** o modelo comete tanto falso positivo (527 casos) quanto falso negativo (386 casos) em quantidade parecida, sem um viés forte pra um lado. Mesmo assim, o falso negativo é o erro mais crítico nesse contexto, porque significa não identificar um estudante que realmente tem indícios de depressão.

- **Que limitações existem:** o dataset depende totalmente de autorrelato (os próprios estudantes respondendo as perguntas), o que pode gerar viés — por exemplo, gente que não relata por causa do estigma. Além disso, o atributo mais forte pra prever (`Have you ever had suicidal thoughts?`) já é, sozinho, um indicador clínico direto de sofrimento psicológico, o que deixa a dúvida se o modelo está realmente "prevendo" depressão ou só captando uma correlação meio óbvia entre dois sintomas relacionados. Um modelo pensado pra uso de verdade precisaria ser testado também sem esse atributo, pra medir o poder preditivo baseado só em fatores de contexto (acadêmicos, financeiros, de rotina).

- **O que dava pra melhorar:** testar o modelo sem o atributo de pensamentos suicidas, pra ver o poder preditivo "de verdade" dos outros fatores; testar técnicas de balanceamento de classes (tipo SMOTE) já que o alvo é meio desbalanceado; fazer uma busca de hiperparâmetros (`GridSearchCV`) pra tentar melhorar mais o RandomForest; e agrupar a coluna `City` em categorias maiores antes do encoding, já que ela gera várias colunas esparsas sem agregar tanto poder preditivo assim.

- **Reflexão ética:** como esse modelo lida com saúde mental, ele não deveria ser usado como ferramenta de diagnóstico sozinho em nenhuma situação real — o objetivo aqui é só acadêmico, estudar se dá pra sinalizar risco a partir de dados de rotina, e qualquer uso prático precisaria de acompanhamento de profissionais de saúde mental e mais cuidado com a privacidade dos dados.
